# MedCLIP-SAMv2 Text+Boxes — Asian MRI Dataset (Water, Lambda)

Applies **MedCLIP-SAMv2** (BiomedCLIP saliency + MuscleMap WB bounding box → MedSAM) to the `MRI_data_asian` Dixon WATER stacks.

Requires MuscleMap WB water segmentations (for bounding boxes) — run `musclemap_asian_water_lambda.ipynb` first.

Structure:
- Images : `~/MRI_data_asian/MRI_data/{01-25}/{Thigh|Calf}/Water.nii.gz`
- MM segs: `~/asian_segs_water/{01-25}/{Thigh|Calf}/*_dseg*`
- Output : `~/medclipsamv2_textboxes_asian_water/{01-25}/{Thigh|Calf}/Water_mcsam2textboxes.npz`

## 1 — Upload to Lambda
```bash
# Asian MRI data
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MRI_data_asian \
  ubuntu@<YOUR-LAMBDA-IP>:~/

# MuscleMap WB asian water segmentations (bounding box source)
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/asian_segs_water/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/asian_segs_water/

# MedSAM checkpoint
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsam_vit_b.pth
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medclipsamv2_textboxes_asian_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2textboxes/asian_segs_water/
```

**Terminate the instance when done.**

In [ ]:
# ── Install SAM in the main Jupyter kernel ────────────────────────────────────
import subprocess, sys

def _ensure(*pkgs):
    import importlib
    missing = [p for p in pkgs
               if importlib.util.find_spec(p.split('[')[0].replace('-','_')) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(missing))
    else:
        print('Already installed:', ', '.join(pkgs))

_ensure('SimpleITK', 'scikit-image')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])

import torch
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# ── Clone MedCLIP-SAMv2 and build venv for BiomedCLIP saliency ───────────────
import os

REPO_DIR = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR = os.path.expanduser('~/mcsam2_env')
VENV_PY  = os.path.join(VENV_DIR, 'bin', 'python')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])
    print('Cloned MedCLIP-SAMv2')
else:
    print('Repo already present')

if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])
    print('Venv created')
else:
    print('Venv already exists')

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')
venv_pip('install', '-q', 'numpy', 'scikit-learn')
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
venv_pip('install', '-q', '-e', os.path.join(REPO_DIR, 'segment-anything'))
venv_pip('install', '-q',
    'git+https://github.com/lucasb-eyer/pydensecrf.git')
venv_pip('install', '-q',
    'open_clip_torch', 'opencv-python', 'SimpleITK', 'Pillow',
    'huggingface_hub', 'transformers<4.46',
    'matplotlib', 'grad-cam', 'pandas', 'tqdm', 'scipy')

print('Venv dependencies installed.')

In [ ]:
# ── Paths and configuration ───────────────────────────────────────────────────
import glob, os, shutil, tempfile
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
import cv2
from PIL import Image
from skimage import transform
from segment_anything import sam_model_registry

REPO_DIR    = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY     = os.path.expanduser('~/mcsam2_env/bin/python')
MEDSAM_CKPT = os.path.expanduser('~/medsam_vit_b.pth')
DATA_ROOT   = os.path.expanduser('~/MRI_data_asian/MRI_data')
MM_SEG_ROOT = os.path.expanduser('~/asian_segs_water')   # MuscleMap WB, 7xxx labels
OUTPUT_ROOT = os.path.expanduser('~/medclipsamv2_textboxes_asian_water')

SAL_SCRIPT  = os.path.join(REPO_DIR, 'saliency_maps', 'generate_saliency_maps.py')
SAM_DEVICE  = 'cpu'
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

for label, path in [
    ('MedSAM ckpt',    MEDSAM_CKPT),
    ('MM asian segs',  MM_SEG_ROOT),
    ('Saliency script', SAL_SCRIPT),
]:
    ok = os.path.exists(path)
    print(f'  {"OK" if ok else "MISSING"}: {label} ({path})')
    if not ok and label in ('MedSAM ckpt', 'MM asian segs'):
        raise FileNotFoundError(f'{label} not found — upload it first')

In [ ]:
# ── Muscle definitions ────────────────────────────────────────────────────────
# (output_key, text_prompt_for_BiomedCLIP, mm_wb_label)
# mm_wb_label: integer label in MuscleMap WB _dseg files (7xxx scheme)
# All 13 bilateral muscle groups from the WB model.
MUSCLES = [
    # ── Quadriceps ──────────────────────────────────────────────────────────
    ('L_vastus_lateralis',
     'vastus lateralis muscle left thigh Dixon MRI axial cross section',
     7101),
    ('R_vastus_lateralis',
     'vastus lateralis muscle right thigh Dixon MRI axial cross section',
     7102),
    ('L_vastus_intermedius',
     'vastus intermedius muscle left thigh Dixon MRI axial cross section',
     7111),
    ('R_vastus_intermedius',
     'vastus intermedius muscle right thigh Dixon MRI axial cross section',
     7112),
    ('L_vastus_medialis',
     'vastus medialis muscle left thigh Dixon MRI axial cross section',
     7121),
    ('R_vastus_medialis',
     'vastus medialis muscle right thigh Dixon MRI axial cross section',
     7122),
    ('L_rectus_femoris',
     'rectus femoris muscle left thigh Dixon MRI axial cross section',
     7131),
    ('R_rectus_femoris',
     'rectus femoris muscle right thigh Dixon MRI axial cross section',
     7132),
    # ── Sartorius / Gracilis ─────────────────────────────────────────────────
    ('L_sartorius',
     'sartorius muscle left thigh Dixon MRI axial cross section',
     7141),
    ('R_sartorius',
     'sartorius muscle right thigh Dixon MRI axial cross section',
     7142),
    ('L_gracilis',
     'gracilis muscle left thigh Dixon MRI axial cross section',
     7151),
    ('R_gracilis',
     'gracilis muscle right thigh Dixon MRI axial cross section',
     7152),
    # ── Hamstrings ───────────────────────────────────────────────────────────
    ('L_semimembranosus',
     'semimembranosus muscle left thigh Dixon MRI axial cross section',
     7161),
    ('R_semimembranosus',
     'semimembranosus muscle right thigh Dixon MRI axial cross section',
     7162),
    ('L_semitendinosus',
     'semitendinosus muscle left thigh Dixon MRI axial cross section',
     7171),
    ('R_semitendinosus',
     'semitendinosus muscle right thigh Dixon MRI axial cross section',
     7172),
    ('L_biceps_femoris_long_head',
     'biceps femoris long head muscle left thigh Dixon MRI axial cross section',
     7181),
    ('R_biceps_femoris_long_head',
     'biceps femoris long head muscle right thigh Dixon MRI axial cross section',
     7182),
    ('L_biceps_femoris_short_head',
     'biceps femoris short head muscle left thigh Dixon MRI axial cross section',
     7191),
    ('R_biceps_femoris_short_head',
     'biceps femoris short head muscle right thigh Dixon MRI axial cross section',
     7192),
    # ── Adductors ────────────────────────────────────────────────────────────
    ('L_adductor_magnus',
     'adductor magnus muscle left thigh Dixon MRI axial cross section',
     7201),
    ('R_adductor_magnus',
     'adductor magnus muscle right thigh Dixon MRI axial cross section',
     7202),
    ('L_adductor_longus',
     'adductor longus muscle left thigh Dixon MRI axial cross section',
     7211),
    ('R_adductor_longus',
     'adductor longus muscle right thigh Dixon MRI axial cross section',
     7212),
    ('L_adductor_brevis',
     'adductor brevis muscle left thigh Dixon MRI axial cross section',
     7221),
    ('R_adductor_brevis',
     'adductor brevis muscle right thigh Dixon MRI axial cross section',
     7222),
]
print(f'{len(MUSCLES)} muscles defined:', [m[0] for m in MUSCLES])

In [ ]:
# ── Discover jobs: (subject, region, water_path, mm_seg_path) ─────────────────
REGIONS = ['Thigh', 'Calf']

jobs = []
for subject in sorted(os.listdir(DATA_ROOT)):
    for region in REGIONS:
        water_path = os.path.join(DATA_ROOT, subject, region, 'Water.nii.gz')
        if not os.path.exists(water_path):
            continue
        mm_candidates = glob.glob(
            os.path.join(MM_SEG_ROOT, subject, region, '*_dseg*')
        )
        if not mm_candidates:
            print(f'  [skip] no MM seg for {subject}/{region}')
            continue
        jobs.append((subject, region, water_path, mm_candidates[0]))

print(f'Found {len(jobs)} jobs')
for subj, reg, _, mm in jobs[:4]:
    print(f'  {subj}/{reg}  MM={os.path.basename(mm)}')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
import glob as _glob
_venv_site = _glob.glob(os.path.join(os.path.expanduser('~/mcsam2_env'),
                                      'lib', 'python3.*', 'site-packages'))
VENV_SITE = _venv_site[0] if _venv_site else ''
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH']       = VENV_SITE + ':' + SUBPROCESS_ENV.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
SUBPROCESS_ENV['MPLBACKEND']       = 'Agg'


def export_slices_as_png(img_array, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        Image.fromarray(np.stack([sl_uint8] * 3, axis=-1)).save(
            os.path.join(out_dir, f'{i}.png')
        )


def run_saliency(png_dir, sal_dir, text_prompt):
    os.makedirs(sal_dir, exist_ok=True)
    result = subprocess.run(
        [VENV_PY, SAL_SCRIPT,
         '--input-path',  png_dir,
         '--output-path', sal_dir,
         '--val-path',    png_dir,
         '--model-name',  'BiomedCLIP',
         '--device',      DEVICE],
        input=text_prompt + '\n',
        text=True, capture_output=True,
        cwd=REPO_DIR, env=SUBPROCESS_ENV,
    )
    if result.returncode != 0:
        print('SALIENCY STDERR:', result.stderr[-1000:])
        raise RuntimeError(f'Saliency failed for prompt: {text_prompt}')


def load_saliency_volume(sal_dir, num_slices, target_size=256):
    vol = np.zeros((num_slices, target_size, target_size), dtype=np.float32)
    for i in range(num_slices):
        png_path = os.path.join(sal_dir, f'{i}.png')
        if os.path.exists(png_path):
            s = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
            if s is not None:
                s = cv2.resize(s, (target_size, target_size),
                               interpolation=cv2.INTER_LINEAR).astype(np.float32)
                vol[i] = (s / 255.0) * 12.0 - 6.0
    return torch.from_numpy(vol[:, None, :, :])


def get_mm_boxes(seg_array, mm_label, img_H, img_W, margin=5):
    D = seg_array.shape[0]
    boxes = []
    for sl in range(D):
        mask = (seg_array[sl] == mm_label).astype(np.uint8)
        if not mask.any():
            boxes.append(None)
            continue
        rows = np.where(np.any(mask, axis=1))[0]
        cols = np.where(np.any(mask, axis=0))[0]
        r0, r1 = rows[[0, -1]]
        c0, c1 = cols[[0, -1]]
        H, W = mask.shape
        box_img = np.array([
            max(0, c0 - margin), max(0, r0 - margin),
            min(W - 1, c1 + margin), min(H - 1, r1 + margin),
        ], dtype=float)
        box_1024 = box_img / np.array([img_W, img_H, img_W, img_H]) * 1024
        boxes.append(box_1024)
    return boxes


def medsam_infer_with_saliency(sam_model, img_embed, box_1024, saliency_256,
                                H, W, device):
    box_t = torch.as_tensor(box_1024, dtype=torch.float, device=device)[None, None, :]
    sal_t = saliency_256.to(device)
    with torch.no_grad():
        sparse_emb, dense_emb = sam_model.prompt_encoder(
            points=None, boxes=box_t, masks=sal_t,
        )
        low_res_logits, _ = sam_model.mask_decoder(
            image_embeddings=img_embed,
            image_pe=sam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
        )
    pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W),
        mode='bilinear', align_corners=False,
    )
    return (pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8)


print('Helpers defined.')

In [ ]:
# ── Load MedSAM once ──────────────────────────────────────────────────────────
sam_model = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT)
sam_model.to(device=SAM_DEVICE)
sam_model.eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
# ── Main processing loop ──────────────────────────────────────────────────────

for subject, region, water_path, mm_seg_path in jobs:
    out_subdir = os.path.join(OUTPUT_ROOT, subject, region)
    out_path   = os.path.join(out_subdir, 'Water_mcsam2textboxes.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {subject}/{region}')
        continue

    print(f'\n═══ {subject}/{region} ═══')
    os.makedirs(out_subdir, exist_ok=True)

    img_array = sitk.GetArrayFromImage(
        sitk.ReadImage(water_path)).astype(float)       # (D, H, W)
    seg_array = sitk.GetArrayFromImage(
        sitk.ReadImage(mm_seg_path))                    # (D, H, W), 7xxx labels
    D, H, W   = img_array.shape
    print(f'  Shape: {img_array.shape}')

    tmp_root  = tempfile.mkdtemp(prefix='mctba_')
    all_masks = {}

    try:
        # ── Export slices once (reused for all muscles) ───────────────────
        png_dir = os.path.join(tmp_root, 'slices')
        export_slices_as_png(img_array, png_dir)

        # ── Precompute SAM image embeddings once per stack ────────────────
        print('  Computing SAM embeddings...')
        embeddings = []
        for sl_idx in range(D):
            sl       = img_array[sl_idx]
            img_norm = sl * 255.0 / (sl.max() + 1e-8)
            img_3c   = np.repeat(img_norm[:, :, None], 3, axis=-1)
            img_1024 = transform.resize(
                img_3c, (1024, 1024), order=3,
                preserve_range=True, anti_aliasing=True,
            ).astype(np.uint8)
            img_1024 = (img_1024 - img_1024.min()) / np.clip(
                img_1024.max() - img_1024.min(), 1e-8, None)
            img_t = (torch.tensor(img_1024).float()
                     .permute(2, 0, 1).unsqueeze(0).to(SAM_DEVICE))
            with torch.no_grad():
                embeddings.append(sam_model.image_encoder(img_t))
            del img_t
        print(f'  {D} embeddings ready')

        # ── Per-muscle: saliency + box → MedSAM ──────────────────────────
        for muscle_name, text_prompt, mm_label in MUSCLES:
            print(f'  [{muscle_name}]')

            sal_dir = os.path.join(tmp_root, f'sal_{muscle_name}')
            run_saliency(png_dir, sal_dir, text_prompt)
            sal_vol = load_saliency_volume(sal_dir, D)      # (D, 1, 256, 256)
            boxes   = get_mm_boxes(seg_array, mm_label, H, W)
            vol_mask = np.zeros((D, H, W), dtype=np.uint8)

            for sl_idx in range(D):
                box = boxes[sl_idx]
                if box is None:
                    # No MM box: threshold saliency directly
                    sal_np = sal_vol[sl_idx, 0].numpy()
                    vol_mask[sl_idx] = cv2.resize(
                        (sal_np > 0).astype(np.uint8), (W, H),
                        interpolation=cv2.INTER_NEAREST,
                    )
                    continue

                vol_mask[sl_idx] = medsam_infer_with_saliency(
                    sam_model, embeddings[sl_idx], box,
                    sal_vol[sl_idx:sl_idx+1], H, W, device=SAM_DEVICE,
                )

            all_masks[muscle_name] = vol_mask
            print(f'    {int(vol_mask.sum()):,} positive voxels')

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved → {out_path}')

    except Exception as exc:
        print(f'  ERROR on {subject}/{region}: {exc}')

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────
results = sorted(glob.glob(os.path.join(OUTPUT_ROOT, '*', '*', '*.npz')))
print(f'Output files found: {len(results)} / {len(jobs)}')
if results:
    sample = np.load(results[0])
    print(f'\nSample: {results[0]}')
    for k in sorted(sample.files):
        arr = sample[k]
        print(f'  {k}: shape={arr.shape}  voxels={int(arr.sum()):,}')